### Create RAG Chain Alternative - Using LCEL (LangChain Expression Language)

In [1]:
import os
# to use our OPENAI API key
from dotenv import load_dotenv 
load_dotenv()

True

In [2]:
# langchain imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
# from langchain.schema import Document - older legacy

# vector store imports
# from langchain_community.vectorstores import Chroma - legacy
from langchain_chroma import Chroma

# utility imports
import numpy as np
from typing import List


##### WE are not creating sample data again , as already done in previous step , and files are available in data directory

###  Document Loading

In [5]:
from langchain_community.document_loaders import DirectoryLoader , TextLoader 

# Load the documents from directory
doc_loader = DirectoryLoader(
    path="data",
    loader_cls=TextLoader,
    loader_kwargs={'encoding':'utf-8'},
    glob="*.txt"

)

documents = doc_loader.load()

print(f"Loaded {len(documents)} documents")
print("Preview of First Doc:\n")
print(documents[0].page_content)

Loaded 3 documents
Preview of First Doc:


    Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    


In [6]:
documents

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    '),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natur

### Document Splitting

In [11]:
# Initializr Text Splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50,
    separators=[" "],
    length_function = len
)

chunks = text_splitter.split_documents(documents)

print(f" Created {len(chunks)} documents from {len(documents)} documents")
print(f"\nChunk Example:")
print(f"\nContent: {chunks[0].page_content}")
print(f"\nMeta Data: {chunks[0].metadata}")

 Created 5 documents from 3 documents

Chunk Example:

Content: Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through

Meta Data: {'source': 'data\\doc_0.txt'}


In [12]:
chunks

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of in

### Embedding Models

In [13]:
os.environ['OPENAI_ENV_KEY'] = os.getenv('OPENAI_API_KEY')

In [14]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x000002641F040980>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x000002641F0412B0>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [15]:
# Create a cromadb vector store
persist_directory = "./chroma_db"

# Initialize chromadb with openAI embeddings
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
    persist_directory=persist_directory,
    collection_name="rag_collection2"
)

print(f" Vector store created with {vectorstore._collection.count()} vectors")
print(f"Persisted to : {persist_directory}")

 Vector store created with 5 vectors
Persisted to : ./chroma_db


### Test Similrity Search

In [16]:
query = "What are the types of machine learning?"

similar_docs = vectorstore.similarity_search(query , k=3)
similar_docs

[Document(id='390fcf9e-697d-4e93-8d36-d4b61a7c2d51', metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(id='051faebf-607a-4fd7-a0c1-5ab85f327d97', metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natu

In [17]:
print(f"Query: {query}")
print(f"\nTop {len(similar_docs)} similar chunks:")
for i, doc in enumerate(similar_docs):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:200] + "...")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")

Query: What are the types of machine learning?

Top 3 similar chunks:

--- Chunk 1 ---
Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are...
Source: data\doc_0.txt

--- Chunk 2 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of i...
Source: data\doc_1.txt

--- Chunk 3 ---
Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recogn...
Source: data\doc_2.txt


In [18]:
# similarity search with score
result_score = vectorstore.similarity_search_with_score(query , k=3)
result_score

[(Document(id='390fcf9e-697d-4e93-8d36-d4b61a7c2d51', metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
  0.7797253131866455),
 (Document(id='051faebf-607a-4fd7-a0c1-5ab85f327d97', metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields l

Initialize LLM, RAG Chain, Prompt Template,Query the RAG system

In [ ]:
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model="gpt-4o-mini-2024-07-18", temperature=0.0 , max_tokens=500)
# # use max_completed_tokens=500 if you want to limit the response length for newer models

In [ ]:
# from langchain.chat_models.base import init_chat_model
# llm = init_chat_model(model="gpt-4o-mini-2024-07-18", temperature=0.0 , max_tokens=500)

In [23]:
from langchain.chat_models.base import init_chat_model

# Explicitly defining the model provider
llm = init_chat_model(model="gpt-4o-mini-2024-07-18", model_provider="openai", temperature=0.0, max_tokens=500)


In [24]:
llm.invoke("What is AI")

AIMessage(content='Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and learn like humans. AI systems can perform tasks that typically require human intelligence, such as understanding natural language, recognizing patterns, solving problems, and making decisions.\n\nAI can be categorized into two main types:\n\n1. **Narrow AI (Weak AI)**: This type of AI is designed to perform a specific task or a narrow range of tasks. Examples include virtual assistants like Siri and Alexa, recommendation systems, and image recognition software.\n\n2. **General AI (Strong AI)**: This type of AI would have the ability to understand, learn, and apply intelligence across a wide range of tasks, similar to a human. General AI remains largely theoretical and has not yet been achieved.\n\nAI technologies include machine learning (where systems learn from data), natural language processing (which enables machines to understand and generate 

### Create RAG Chain Alternative - Using LCEL (LangChain Expression Language)

In [25]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

In [26]:
# Create a custom prompt
from langchain_core.prompts import ChatPromptTemplate
custom_prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question. 
If you don't know the answer based on the context, say you don't know.
Provide specific details from the context to support your answer.

Context:
{context}

Question: {question}

Answer:""")
custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question. \nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])

In [28]:
## Convert vector store to retriever
retriever=vectorstore.as_retriever(
    search_kwargs={"k":3} ## Retrieve top 3 relevant chunks
)
retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000002641F041400>, search_kwargs={'k': 3})

In [29]:
## Format the output documents for the prompt
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [30]:
## Build the chain ussing LCEL

rag_chain_lcel=(
    { 
        "context":retriever | format_docs,
        "question": RunnablePassthrough()
     }
    | custom_prompt
    | llm
    | StrOutputParser()
)

rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000002641F041400>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question. \nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])
| ChatOpenAI(output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x000002641F02E8B0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000

In [31]:
response=rag_chain_lcel.invoke("What is Deep Learning")
response

'Deep learning is a subset of machine learning based on artificial neural networks. These networks are inspired by the human brain and consist of layers of interconnected nodes. Deep learning has significantly impacted various fields, including computer vision, natural language processing, and speech recognition.'

In [34]:
# retriever.get_relevant_documents("What is Deep Learning") 
# depeceated

In [33]:
# The modern way to fetch relevant chunks manually
retriever.invoke("What is Deep Learning")

[Document(id='051faebf-607a-4fd7-a0c1-5ab85f327d97', metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers'),
 Document(id='390fcf9e-697d-4e93-8d36-d4b61a7c2d51', metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised le

In [36]:
# Query using the LCEL approach - Fixed version
def query_rag_lcel(question):
    print(f"Question: {question}")
    print("-" * 50)
    
    # Method 1: Pass string directly (when using RunnablePassthrough)
    answer = rag_chain_lcel.invoke(question)
    print(f"Answer: {answer}")
    
    # Get source documents separately if needed
    # docs = retriever.get_relevant_documents(question) -- deprecated
    docs = retriever.invoke(question)

    print("\nSource Documents:")
    for i, doc in enumerate(docs):
        print(f"\n--- Source {i+1} ---")
        print(doc.page_content[:200] + "...")

In [37]:
# Test LCEL chain
print("Testing LCEL Chain:")
query_rag_lcel("What are the key concepts in reinforcement learning?")

Testing LCEL Chain:
Question: What are the key concepts in reinforcement learning?
--------------------------------------------------
Answer: The key concepts in reinforcement learning, as indicated in the context, include learning through interaction with an environment and the use of rewards and penalties. Reinforcement learning focuses on how agents take actions in an environment to maximize cumulative rewards, learning from the consequences of their actions rather than from labeled data, as seen in supervised learning.

Source Documents:

--- Source 1 ---
data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties....

--- Source 2 ---
Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are...

--- Source 3 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learn

In [38]:
query_rag_lcel("What is machine learning?")

Question: What is machine learning?
--------------------------------------------------
Answer: Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It involves three main types: supervised learning, which uses labeled data to train models; unsupervised learning, which finds patterns in unlabeled data; and reinforcement learning, which learns through interactions with an environment.

Source Documents:

--- Source 1 ---
Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are...

--- Source 2 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of i...

--- Source 3 ---
Natural Language Processing (NLP)



In [39]:
query_rag_lcel("What is CNN?")

Question: What is CNN?
--------------------------------------------------
Answer: CNN stands for Convolutional Neural Network. It is a type of deep learning architecture that is particularly effective for image processing. CNNs are designed to automatically and adaptively learn spatial hierarchies of features from images, making them well-suited for tasks in computer vision.

Source Documents:

--- Source 1 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of i...

--- Source 2 ---
Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recogn...

--- Source 3 ---
Neural Networks (RNNs) and Transformers 
    excel at sequential data processing....


In [40]:
query_rag_lcel("What is gradient descent?")

Question: What is gradient descent?
--------------------------------------------------
Answer: I don't know. The provided context does not mention gradient descent or provide any information related to it.

Source Documents:

--- Source 1 ---
data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties....

--- Source 2 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of i...

--- Source 3 ---
Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are...
